In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

In [21]:
os.makedirs("plots/eda", exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 130
 
df = pd.read_csv("tshirts_preprocessed.csv", parse_dates=["week_start"])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}\n")

Loaded: 834,538 rows × 112 columns
Columns: ['article_id', 'week_start', 'weekly_sales_volume', 'avg_weekly_price', 'year_month', 'product_type_name', 'product_group_name', 'week_of_year', 'month', 'quarter', 'year', 'is_summer', 'product_age_weeks', 'sales_lag1', 'sales_lag2', 'sales_rolling4', 'real_price', 'eurozone_hicp', 'eurozone_unemployment_rate', 'eurozone_cci', 'is_spring_summer', 'is_sale_season', 'sales_lag4', 'sales_rolling4_mean', 'sales_rolling4_std', 'price_vs_median', 'hicp_lag1m', 'unemployment_lag1m', 'cci_lag1m', 'graphical_appearance_name_Application/3D', 'graphical_appearance_name_Argyle', 'graphical_appearance_name_Check', 'graphical_appearance_name_Colour blocking', 'graphical_appearance_name_Contrast', 'graphical_appearance_name_Dot', 'graphical_appearance_name_Embroidery', 'graphical_appearance_name_Front print', 'graphical_appearance_name_Glittering/Metallic', 'graphical_appearance_name_Jacquard', 'graphical_appearance_name_Lace', 'graphical_appearance_name_M

In [22]:
REFERENCE_CATEGORIES = {
    "index_group_name":            "Baby/Children",
    "colour_group_name":           "Black",
    "graphical_appearance_name":   "Melange",
    "perceived_colour_value_name": "Dark",
}

In [23]:
def reconstruct_category(df, prefix):
    """
    Reconstruct the original categorical column from its one-hot encoded columns.
    Returns a Series with the label for each row.
    """
    ohe_cols = sorted([c for c in df.columns if c.startswith(prefix + "_")])
 
    if not ohe_cols:
        # Column was never OHE'd — return it as-is if it still exists
        if prefix in df.columns:
            return df[prefix].copy()
        print(f"  WARNING: no columns found for prefix '{prefix}'")
        return None
 
    reference = REFERENCE_CATEGORIES.get(prefix, "Reference_category")
 
    # Start with reference label for every row
    result = pd.Series(reference, index=df.index, name=prefix)
 
    # For each OHE column, assign its label where it is 1
    for col in ohe_cols:
        label = col[len(prefix) + 1:]          # strip "prefix_"
        result[df[col] == 1] = label
 
    # Quick sanity check
    n_reference = (df[ohe_cols].sum(axis=1) == 0).sum()
    labels_found = sorted(result.unique())
    print(f"  {prefix}: {labels_found}  ({n_reference:,} rows = '{reference}')")
    return result

In [24]:
print("Reconstructing categorical columns from OHE:")
df["index_group_name"]            = reconstruct_category(df, "index_group_name")
df["colour_group_name"]           = reconstruct_category(df, "colour_group_name")
df["graphical_appearance_name"]   = reconstruct_category(df, "graphical_appearance_name")
df["perceived_colour_value_name"] = reconstruct_category(df, "perceived_colour_value_name")
print()

Reconstructing categorical columns from OHE:
  index_group_name: ['Baby/Children', 'Divided', 'Ladieswear', 'Menswear', 'Sport']  (357,962 rows = 'Baby/Children')
  colour_group_name: ['Black', 'Blue', 'Dark Beige', 'Dark Blue', 'Dark Green', 'Dark Grey', 'Dark Orange', 'Dark Pink', 'Dark Purple', 'Dark Red', 'Dark Turquoise', 'Dark Yellow', 'Green', 'Greenish Khaki', 'Grey', 'Greyish Beige', 'Light Beige', 'Light Blue', 'Light Green', 'Light Grey', 'Light Orange', 'Light Pink', 'Light Purple', 'Light Red', 'Light Turquoise', 'Light Yellow', 'Off White', 'Orange', 'Other', 'Other Blue', 'Other Green', 'Other Orange', 'Other Pink', 'Other Purple', 'Other Red', 'Other Yellow', 'Pink', 'Purple', 'Red', 'Silver', 'Turquoise', 'Unknown', 'White', 'Yellow', 'Yellowish Brown']  (11,766 rows = 'Black')


/var/folders/y6/ns1t9zy17qngjlq4qm24kjy00000gn/T/ipykernel_8839/3044349693.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["index_group_name"]            = reconstruct_category(df, "index_group_name")
/var/folders/y6/ns1t9zy17qngjlq4qm24kjy00000gn/T/ipykernel_8839/3044349693.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["colour_group_name"]           = reconstruct_category(df, "colour_group_name")
/var/folders/y6/ns1t9zy17qngjlq4qm24kjy00000gn/T/ipykernel_8839/3044349693.py:4: PerformanceWarning: DataFrame is hig

  graphical_appearance_name: ['Application/3D', 'Argyle', 'Check', 'Colour blocking', 'Contrast', 'Dot', 'Embroidery', 'Front print', 'Glittering/Metallic', 'Jacquard', 'Lace', 'Melange', 'Mesh', 'Metallic', 'Mixed solid/pattern', 'Neps', 'Other pattern', 'Other structure', 'Placement print', 'Sequin', 'Slub', 'Solid', 'Stripe', 'Transparent', 'Treatment', 'Unknown']  (100,806 rows = 'Melange')
  perceived_colour_value_name: ['Dark', 'Dusty Light', 'Light', 'Medium', 'Medium Dusty', 'Undefined', 'Unknown']  (49,926 rows = 'Dark')



/var/folders/y6/ns1t9zy17qngjlq4qm24kjy00000gn/T/ipykernel_8839/3044349693.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["perceived_colour_value_name"] = reconstruct_category(df, "perceived_colour_value_name")


In [25]:
weekly_total = df.groupby("week_start")["weekly_sales_volume"].sum().reset_index()
 
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(weekly_total["week_start"], weekly_total["weekly_sales_volume"],
                alpha=0.25, color="steelblue")
ax.plot(weekly_total["week_start"], weekly_total["weekly_sales_volume"],
        color="steelblue", linewidth=1.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title("Total Weekly T-Shirt Sales Volume Over Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Week")
ax.set_ylabel("Units Sold")
plt.tight_layout()
plt.savefig("plots/eda/A1_weekly_total_sales.png")
plt.close()
print("Saved: A1_weekly_total_sales.png")

Saved: A1_weekly_total_sales.png


In [26]:
seg_weekly = (
    df.groupby(["week_start", "index_group_name"])["weekly_sales_volume"]
    .sum()
    .reset_index()
)
 
fig, ax = plt.subplots(figsize=(13, 4))
palette = sns.color_palette("tab10", n_colors=seg_weekly["index_group_name"].nunique())
for (seg, grp), color in zip(seg_weekly.groupby("index_group_name"), palette):
    ax.plot(grp["week_start"], grp["weekly_sales_volume"],
            label=seg, linewidth=1.8, color=color)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title("Weekly T-Shirt Sales by Customer Segment", fontsize=14, fontweight="bold")
ax.set_xlabel("Week")
ax.set_ylabel("Units Sold")
ax.legend(title="Segment")
plt.tight_layout()
plt.savefig("plots/eda/A2_sales_by_segment.png")
plt.close()
print("Saved: A2_sales_by_segment.png")

Saved: A2_sales_by_segment.png


In [28]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
def horizontal_bar(ax, col, title, top_n=10, color="steelblue"):
    counts = (
        df.groupby(col)["weekly_sales_volume"]
        .sum()
        .nlargest(top_n)
        .sort_values()
    )
    counts.plot(kind="barh", ax=ax, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Total Units Sold")

horizontal_bar(axes[0], "index_group_name",          "Sales by Customer Segment",  color="steelblue")
horizontal_bar(axes[1], "colour_group_name",          "Sales by Colour Group",      color="coral")
horizontal_bar(axes[2], "graphical_appearance_name",  "Sales by Appearance Style",  color="mediumseagreen")
 
plt.suptitle("T-Shirt Sales Distribution by Product Attributes",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/eda/B1_product_attribute_breakdown.png")
plt.close()
print("Saved: B1_product_attribute_breakdown.png")

Saved: B1_product_attribute_breakdown.png


In [29]:
fig, ax = plt.subplots(figsize=(8, 4))
colour_val = (
    df.groupby("perceived_colour_value_name")["weekly_sales_volume"]
    .sum()
    .sort_values(ascending=True)
)
colour_val.plot(kind="barh", ax=ax, color="orchid", edgecolor="white", alpha=0.85)
ax.set_title("Sales by Perceived Colour Value", fontweight="bold")
ax.set_xlabel("Total Units Sold")
plt.tight_layout()
plt.savefig("plots/eda/B2_colour_value_breakdown.png")
plt.close()
print("Saved: B2_colour_value_breakdown.png")

Saved: B2_colour_value_breakdown.png


In [30]:
age_sales = (
    df[df["product_age_weeks"] >= 0]
    .groupby("product_age_weeks")["weekly_sales_volume"]
    .agg(mean="mean", median="median", count="count")
    .reset_index()
)
age_sales = age_sales[age_sales["count"] >= 20] 

In [31]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(age_sales["product_age_weeks"], age_sales["mean"],
        label="Mean sales", color="steelblue", linewidth=2)
ax.plot(age_sales["product_age_weeks"], age_sales["median"],
        label="Median sales", color="coral", linewidth=2, linestyle="--")
ax.fill_between(age_sales["product_age_weeks"],
                age_sales["mean"] - age_sales["mean"].std(),
                age_sales["mean"] + age_sales["mean"].std(),
                alpha=0.1, color="steelblue", label="±1 std")
ax.set_title("Average Weekly Sales vs. Product Age (weeks since first sale)\n"
             "Based on Vashishtha et al. (2020) product lifecycle methodology",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Product Age (weeks)")
ax.set_ylabel("Weekly Sales Volume")
ax.legend()
plt.tight_layout()
plt.savefig("plots/eda/C1_product_lifecycle.png")
plt.close()
print("Saved: C1_product_lifecycle.png")

Saved: C1_product_lifecycle.png


In [32]:
active_per_week = (
    df[df["weekly_sales_volume"] > 0]
    .groupby("week_start")["article_id"]
    .nunique()
    .reset_index()
    .rename(columns={"article_id": "active_articles"})
)
 
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(active_per_week["week_start"], active_per_week["active_articles"],
        color="mediumseagreen", linewidth=1.8)
ax.fill_between(active_per_week["week_start"], active_per_week["active_articles"],
                alpha=0.15, color="mediumseagreen")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title("Number of Actively Selling T-Shirt Articles per Week",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Week")
ax.set_ylabel("Active Articles")
plt.tight_layout()
plt.savefig("plots/eda/C2_active_articles_over_time.png")
plt.close()
print("Saved: C2_active_articles_over_time.png")

Saved: C2_active_articles_over_time.png


In [33]:
macro_monthly = (
    df.groupby("year_month")[
        ["eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci"]
    ]
    .mean()
    .reset_index()
    .sort_values("year_month")
)
macro_monthly["date"] = pd.to_datetime(macro_monthly["year_month"])

In [34]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
specs = [
    ("eurozone_hicp",              "HICP Inflation Rate (%)",   "darkorange"),
    ("eurozone_unemployment_rate", "Unemployment Rate (%)",     "tomato"),
    ("eurozone_cci",               "Consumer Confidence Index", "steelblue"),
]
for ax, (col, label, color) in zip(axes, specs):
    ax.plot(macro_monthly["date"], macro_monthly[col], color=color, linewidth=2)
    ax.fill_between(macro_monthly["date"], macro_monthly[col],
                    alpha=0.15, color=color)
    ax.set_ylabel(label, fontsize=10)
    ax.axvline(pd.Timestamp("2020-03-01"), color="crimson", linestyle="--",
               linewidth=1.2, label="COVID-19 (Mar 2020)")
    ax.legend(fontsize=9, loc="upper left")
 
axes[0].set_title("Eurozone Macroeconomic Indicators Over Study Period\n(HICP used to compute inflation-adjusted real prices)",
                  fontsize=14, fontweight="bold")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("plots/eda/D1_macro_indicators.png")
plt.close()
print("Saved: D1_macro_indicators.png")

Saved: D1_macro_indicators.png


In [35]:
weekly_agg = df.groupby("week_start").agg(
    total_sales=("weekly_sales_volume", "sum"),
    cci=("eurozone_cci", "mean")
).reset_index()
 
fig, ax1 = plt.subplots(figsize=(13, 4))
ax2 = ax1.twinx()
ax1.fill_between(weekly_agg["week_start"], weekly_agg["total_sales"],
                 alpha=0.2, color="steelblue")
ax1.plot(weekly_agg["week_start"], weekly_agg["total_sales"],
         color="steelblue", linewidth=1.8, label="Weekly Sales")
ax2.plot(weekly_agg["week_start"], weekly_agg["cci"],
         color="darkorange", linewidth=2, linestyle="--", label="CCI")
ax1.set_ylabel("Total Weekly Sales", color="steelblue")
ax2.set_ylabel("Consumer Confidence Index", color="darkorange")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax1.set_title("Weekly T-Shirt Sales vs. Consumer Confidence Index",
              fontsize=14, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.savefig("plots/eda/D2_sales_vs_cci.png")
plt.close()
print("Saved: D2_sales_vs_cci.png")

Saved: D2_sales_vs_cci.png


In [36]:
weekly_unemp = df.groupby("week_start").agg(
    total_sales=("weekly_sales_volume", "sum"),
    unemp=("eurozone_unemployment_rate", "mean")
).reset_index()
 
fig, ax1 = plt.subplots(figsize=(13, 4))
ax2 = ax1.twinx()
ax1.fill_between(weekly_unemp["week_start"], weekly_unemp["total_sales"],
                 alpha=0.2, color="steelblue")
ax1.plot(weekly_unemp["week_start"], weekly_unemp["total_sales"],
         color="steelblue", linewidth=1.8, label="Weekly Sales")
ax2.plot(weekly_unemp["week_start"], weekly_unemp["unemp"],
         color="tomato", linewidth=2, linestyle="--", label="Unemployment Rate")
ax1.set_ylabel("Total Weekly Sales", color="steelblue")
ax2.set_ylabel("Unemployment Rate (%)", color="tomato")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2.tick_params(axis="y", labelcolor="tomato")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax1.set_title("Weekly T-Shirt Sales vs. Unemployment Rate",
              fontsize=14, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.savefig("plots/eda/D3_sales_vs_unemployment.png")
plt.close()
print("Saved: D3_sales_vs_unemployment.png")

Saved: D3_sales_vs_unemployment.png


In [37]:
zero_pct_per_article = (
    df.groupby("article_id")["weekly_sales_volume"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index()
    .rename(columns={"weekly_sales_volume": "pct_zero_weeks"})
)

In [39]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
# E1a — Distribution of zero-sale % per article
axes[0].hist(zero_pct_per_article["pct_zero_weeks"], bins=30,
             color="steelblue", edgecolor="white", alpha=0.85)
median_zero = zero_pct_per_article["pct_zero_weeks"].median()
axes[0].axvline(median_zero, color="tomato", linestyle="--", linewidth=1.5,
                label=f"Median = {median_zero:.1f}%")
axes[0].set_title("Zero-Sale Week % per Article", fontweight="bold")
axes[0].set_xlabel("% of Weeks with Zero Sales")
axes[0].set_ylabel("Article Count")
axes[0].legend()
 
# E1b — Sales volume distribution (log-transformed, non-zero only)
nonzero_sales = df[df["weekly_sales_volume"] > 0]["weekly_sales_volume"]
axes[1].hist(np.log1p(nonzero_sales), bins=50, color="coral",
             edgecolor="white", alpha=0.85)
axes[1].set_title("Distribution of log(1 + Weekly Sales)\n(non-zero weeks only)",
                  fontweight="bold")
axes[1].set_xlabel("log(1 + Sales Volume)")
axes[1].set_ylabel("Count")
 
pct_zero_overall = (df["weekly_sales_volume"] == 0).mean() * 100
plt.suptitle(f"Intermittent Demand Patterns  —  {pct_zero_overall:.1f}% of all article-weeks have zero sales",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/eda/E1_intermittent_demand.png")
plt.close()
print("Saved: E1_intermittent_demand.png")

Saved: E1_intermittent_demand.png


In [40]:
heatmap_data = (
    df.groupby(["year", "week_of_year"])["weekly_sales_volume"]
    .mean()
    .unstack(level=0)
)

In [41]:
fig, ax = plt.subplots(figsize=(18, 4))
sns.heatmap(
    heatmap_data.T,
    cmap="YlOrRd",
    linewidths=0.3,
    cbar_kws={"label": "Mean Weekly Sales"},
    ax=ax
)
ax.set_title("Seasonality Heatmap — Mean Weekly T-Shirt Sales by Year × Week of Year",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Week of Year")
ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig("plots/eda/F1_seasonality_heatmap.png")
plt.close()
print("Saved: F1_seasonality_heatmap.png")

Saved: F1_seasonality_heatmap.png


In [ ]:
# Price trend: nominal vs. inflation-adjusted (real_price)
# Note: real_price = avg_weekly_price / (1 + HICP/100) per supervisor's instruction
price_weekly = (
    df[df['weekly_sales_volume'] > 0]
    .groupby('week_start')
    .agg(avg_price=('avg_weekly_price', 'mean'),
         real_price=('real_price', 'mean'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(price_weekly['week_start'], price_weekly['avg_price'],
        color='steelblue', linewidth=1.8, label='Nominal price')
ax.plot(price_weekly['week_start'], price_weekly['real_price'],
        color='coral', linewidth=1.8, linestyle='--', label='Real price (inflation-adjusted, HICP)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title('Average Weekly T-Shirt Price: Nominal vs. Inflation-Adjusted\n'
             '(Inflation-adjusted = nominal price deflated by Eurozone HICP)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Average Price (EUR)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/eda/G1_price_nominal_vs_real.png')
plt.close()
print('Saved: G1_price_nominal_vs_real.png')

In [42]:
print("\n" + "=" * 55)
print("EDA COMPLETE — plots saved to plots/eda/")
print("=" * 55)
print(f"  A1 Total weekly sales over time")
print(f"  A2 Sales by customer segment         ← was broken, now fixed")
print(f"  B1 Product attribute breakdown        ← was broken, now fixed")
print(f"  B2 Colour value breakdown             ← was broken, now fixed")
print(f"  C1 Product lifecycle curve")
print(f"  C2 Active articles over time")
print(f"  D1 Three macro indicators stacked")
print(f"  D2 Sales vs. CCI overlay")
print(f"  D3 Sales vs. Unemployment overlay")
print(f"  E1 Intermittent demand patterns")
print(f"  F1 Seasonality heatmap")


EDA COMPLETE — plots saved to plots/eda/
  A1 Total weekly sales over time
  A2 Sales by customer segment         ← was broken, now fixed
  B1 Product attribute breakdown        ← was broken, now fixed
  B2 Colour value breakdown             ← was broken, now fixed
  C1 Product lifecycle curve
  C2 Active articles over time
  D1 Three macro indicators stacked
  D2 Sales vs. CCI overlay
  D3 Sales vs. Unemployment overlay
  E1 Intermittent demand patterns
  F1 Seasonality heatmap
